<a href="https://colab.research.google.com/github/Shaithra2107/HALT-RAG/blob/lift_signal/HALT_RAG_corpus_validate_v5_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HALT-RAG · Corpus Validation v4

Three metrics, each explicitly scoped:

| # | Metric | What it actually measures |
|---|--------|--------------------------|
| **M1** | Subject-page match (Wikipedia now) | Answer string in current Wikipedia article for subject entity |
| **M2** | Subject-page match (KILT 2019) | Answer string in KILT 2019 page for subject entity, within scan window |
| **M3** | Pilot retrieval@5 (small index) | Answer string in FAISS top-5 passages, convenience-sampled pilot index |

**Scope limits (must be stated in any report):**
- M2 covers only pages whose normalised title matched within `kilt_scan_limit` pages.
  Pages not reached are reported as **unknown**, not as confirmed negatives.
- M3 uses a pilot FAISS index (matched entity pages + convenience-sampled distractors).
  It is a smoke test, not a benchmark of full-KILT retrieval.
- Answer-string matching (M1/M2/M3) is not the same as semantic answerability.


In [4]:
# datasets>=3.0 removed support for legacy dataset scripts; facebook/kilt_wikipedia
# still uses one, so we must pin to 2.x.
!pip install -q "datasets>=2.14,<3.0" wikipedia-api sentence-transformers faiss-cpu tqdm pandas numpy
# Record versions for reproducibility
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'show',
                         'datasets', 'sentence-transformers', 'faiss-cpu'],
                        capture_output=True, text=True)
print(result.stdout)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
Name: datasets
Version: 2.21.0
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: aiohttp, dill, filelock, fsspec, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: torchtune
---
Name: sentence-transformers
Version: 5.7.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License-Expression: Apache-2.0
Location: /usr/local/lib/py

In [5]:
import pandas as pd
import numpy as np
import re, time, ast, json
from datetime import datetime
from datasets import load_dataset
import wikipediaapi
from sentence_transformers import SentenceTransformer
import faiss
from tqdm.auto import tqdm

CONFIG = {
    'n_tail'         : 150,
    'n_head'         : 150,
    'tail_max_pop'   : 50,
    'head_min_pop'   : 500,
    'seed'           : 42,
    'top_k'          : 5,
    'chunk_words'    : 100,
    'n_distractors'  : 500,
    'kilt_scan_limit'  : 3_000_000,
    'checkpoint_every' : 200_000,   # save partial progress every N records
    'min_ans_chars'    : 2,         # filter answers shorter than this
    'run_timestamp'  : datetime.utcnow().isoformat() + 'Z',
}
print("Config ready:", json.dumps(CONFIG, indent=2))

Config ready: {
  "n_tail": 150,
  "n_head": 150,
  "tail_max_pop": 50,
  "head_min_pop": 500,
  "seed": 42,
  "top_k": 5,
  "chunk_words": 100,
  "n_distractors": 500,
  "kilt_scan_limit": 3000000,
  "checkpoint_every": 200000,
  "min_ans_chars": 2,
  "run_timestamp": "2026-10-24T17:58:55.228634Z"
}


/tmp/ipykernel_1636/3734624268.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'run_timestamp'  : datetime.utcnow().isoformat() + 'Z',


In [6]:
def normalise_text(text):
    """Lowercase + collapse whitespace for matching."""
    return re.sub(r'\s+', ' ', str(text).lower().strip())

def normalise_title(title):
    """Normalise a Wikipedia title for lookup: lowercase, underscores→spaces."""
    return re.sub(r'_', ' ', str(title).lower().strip())

def get_title(row):
    """Return the best Wikipedia title for this row, never NaN/None."""
    t = row['s_wiki_title']
    if t and isinstance(t, str) and t.strip():
        return t.strip()
    return str(row['subj']).strip()

def parse_answers(val):
    """Parse possible_answers to a clean list, filtering empty/NaN."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    if isinstance(val, list):
        ans = val
    else:
        try:    ans = ast.literal_eval(str(val))
        except: ans = [str(val)]
    # filter empty, NaN strings, and very short answers
    cleaned = []
    for a in ans:
        s = str(a).strip()
        if s and s.lower() != 'nan' and len(s) >= CONFIG['min_ans_chars']:
            cleaned.append(s)
    return cleaned

def answer_in_text(answers, text_normalised):
    """Check if any answer appears as a word-boundary match."""
    for ans in answers:
        a = normalise_text(ans)
        if not a:
            continue
        # word-boundary: ensure match isn't inside a longer token
        pattern = r'(?<![\w])' + re.escape(a) + r'(?![\w])'
        if re.search(pattern, text_normalised):
            return True
    return False

In [7]:
raw    = load_dataset("akariasai/PopQA", split="test")
df_all = raw.to_pandas()
print(f"PopQA: {len(df_all):,} rows  |  columns: {df_all.columns.tolist()}")

Repo card metadata block was not found. Setting CardData to empty.


Generating test split:   0%|          | 0/14267 [00:00<?, ? examples/s]

PopQA: 14,267 rows  |  columns: ['id', 'subj', 'prop', 'obj', 'subj_id', 'prop_id', 'obj_id', 's_aliases', 'o_aliases', 's_uri', 'o_uri', 's_wiki_title', 'o_wiki_title', 's_pop', 'o_pop', 'question', 'possible_answers']


In [8]:
tail_pool = df_all[df_all['s_pop'] < CONFIG['tail_max_pop']]
head_pool = df_all[df_all['s_pop'] > CONFIG['head_min_pop']]

n_t = min(CONFIG['n_tail'], len(tail_pool))
n_h = min(CONFIG['n_head'], len(head_pool))

df_tail = tail_pool.sample(n=n_t, random_state=CONFIG['seed']).copy()
df_head = head_pool.sample(n=n_h, random_state=CONFIG['seed']).copy()
df_tail['band'] = 'tail'
df_head['band'] = 'head'

df = pd.concat([df_tail, df_head]).reset_index(drop=True)
df['answers_list'] = df['possible_answers'].apply(parse_answers)
df['wiki_title']   = df.apply(get_title, axis=1)

print(f"Sample: {n_t} tail + {n_h} head = {len(df)} total")
empty_ans = (df['answers_list'].apply(len) == 0).sum()
print(f"Rows with empty answer list: {empty_ans}")
print(df.groupby('band')['s_pop'].describe()[['min','mean','max']].to_string())

Sample: 150 tail + 150 head = 300 total
Rows with empty answer list: 0
        min          mean       max
band                               
head  502.0  23317.880000  420204.0
tail    2.0     34.106667      49.0


## M1 — Subject-page match (current Wikipedia)
Sanity check only. Not KILT evidence.


In [9]:
wiki = wikipediaapi.Wikipedia(
    user_agent='HALT-RAG-Research/1.0', language='en')

m1_match, m1_status = [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc='M1 Wikipedia'):
    time.sleep(0.25)
    title = row['wiki_title']
    page  = wiki.page(title)
    if not page.exists():
        m1_match.append(0); m1_status.append('article_missing')
        continue
    body = normalise_text(page.text)
    hit  = answer_in_text(row['answers_list'], body)
    m1_match.append(int(hit))
    m1_status.append('found' if hit else 'not_found')

df['m1_match']  = m1_match
df['m1_status'] = m1_status

print("M1 done.")
print(df['m1_status'].value_counts().to_string())

M1 Wikipedia:   0%|          | 0/300 [00:00<?, ?it/s]

M1 done.
m1_status
found              263
not_found           34
article_missing      3


In [10]:
print("=" * 60)
print("M1: SUBJECT-PAGE MATCH (current Wikipedia)")
print("=" * 60)
for band in ['tail','head','overall']:
    sub = df[df['band']==band] if band!='overall' else df
    found  = (sub['m1_status']=='found').sum()
    miss   = (sub['m1_status']=='article_missing').sum()
    total  = len(sub)
    print(f"  {band:7s}: {found/total:.1%} match  "
          f"({found}/{total})  missing_article={miss}")

M1: SUBJECT-PAGE MATCH (current Wikipedia)
  tail   : 80.7% match  (121/150)  missing_article=3
  head   : 94.7% match  (142/150)  missing_article=0
  overall: 87.7% match  (263/300)  missing_article=3


## KILT load test
**Run this cell alone first.** If it fails, fix the environment before launching the full scan.


In [11]:
print("Testing KILT load (1 record)...")
kilt_test = load_dataset("facebook/kilt_wikipedia", split="full",
                          streaming=True, trust_remote_code=True)
rec = next(iter(kilt_test))
print("Keys      :", list(rec.keys()))
print("Title     :", rec['wikipedia_title'])
paras = rec['text']['paragraph'] if isinstance(rec['text'], dict) else []
print(f"Paragraphs: {len(paras)}")
for p in paras[:2]:
    print(" ", p[:120])
print("\n✓ KILT dataset loads correctly. Proceed to full scan.")

Testing KILT load (1 record)...


Keys      : ['kilt_id', 'wikipedia_id', 'wikipedia_title', 'text', 'anchors', 'categories', 'wikidata_info', 'history']
Title     : A
Paragraphs: 69
  A

  A (named , plural "As", "A's", "a"s, "a's" or "aes") is the first letter and the first vowel of the modern English alpha

✓ KILT dataset loads correctly. Proceed to full scan.


In [12]:
# Optional: mount Google Drive so the KILT scan cache survives runtime resets.
# Skip this cell if you don't want to use Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except Exception as e:
    print(f"Drive not mounted ({e}). Cache will be local only.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## M2 — KILT subject-page match (2019 snapshot)
Streams KILT. Title matching is normalised (lower-case, underscores → spaces).
Pages not reached within the scan limit are labelled **unknown** (not confirmed negative).

First run: ~15-30 min in Colab. Checkpoints every 200k records — **safe to disconnect and resume.**


In [13]:
import pickle, hashlib, os
from itertools import islice

# ── Persistent cache dir: Google Drive if mounted, else local ─────────────
_drive_dir = '/content/drive/MyDrive/halt_rag_cache'
_local_dir = '/content/halt_rag_cache'
if os.path.isdir('/content/drive/MyDrive'):
    CACHE_DIR = _drive_dir
    print("Google Drive mounted — checkpoints/cache will survive runtime resets.")
else:
    CACHE_DIR = _local_dir
    print("Drive NOT mounted — files are local (lost on reset). Run Cell 8a to mount Drive.")
os.makedirs(CACHE_DIR, exist_ok=True)

# Build normalised title -> original title lookup
target_norm2orig = {}
for _, row in df.iterrows():
    norm = normalise_title(row['wiki_title'])
    target_norm2orig[norm] = row['wiki_title']
print(f"Target titles to find: {len(target_norm2orig)}")

# Cache key: hash of (scan_limit + sorted target titles)
_sig = hashlib.md5(
    (str(CONFIG['kilt_scan_limit']) + '|' + '|'.join(sorted(target_norm2orig.keys()))).encode()
).hexdigest()[:12]
KILT_CACHE      = os.path.join(CACHE_DIR, f"kilt_cache_{_sig}.pkl")
KILT_CHECKPOINT = os.path.join(CACHE_DIR, f"kilt_ckpt_{_sig}.pkl")

# ── 1. Completed cache? ───────────────────────────────────────────────────
if os.path.exists(KILT_CACHE):
    print(f"\nLoading completed cache: {KILT_CACHE}")
    with open(KILT_CACHE, 'rb') as f:
        _c = pickle.load(f)
    kilt_pages, distractor_pool, scanned = _c['kilt_pages'], _c['distractor_pool'], _c['scanned']
    print(f"Loaded {len(kilt_pages)} entity pages + {len(distractor_pool)} distractors (scanned={scanned:,})")

else:
    # ── 2. Partial checkpoint? ────────────────────────────────────────────
    kilt_pages, distractor_pool, scanned, iter_state = {}, [], 0, None
    if os.path.exists(KILT_CHECKPOINT):
        print(f"\nFound checkpoint: {KILT_CHECKPOINT}")
        with open(KILT_CHECKPOINT, 'rb') as f:
            _c = pickle.load(f)
        kilt_pages      = _c['kilt_pages']
        distractor_pool = _c['distractor_pool']
        scanned         = _c['scanned']
        iter_state      = _c.get('iter_state')
        print(f"Resuming from {scanned:,} records ({len(kilt_pages)} pages found so far)")
    else:
        print(f"\nNo cache or checkpoint — starting fresh scan (~15-30 min)...")

    # ── 3. Build iterator, restore state or fast-forward ─────────────────
    kilt_ds = load_dataset("facebook/kilt_wikipedia", split="full",
                            streaming=True, trust_remote_code=True)
    kilt_it = iter(kilt_ds)

    if iter_state is not None:
        try:
            kilt_it.load_state_dict(iter_state)
            print(f"Streaming iterator state restored — resuming exactly at record {scanned:,}.")
        except Exception as e:
            print(f"state_dict restore failed ({e}). Fast-forwarding {scanned:,} records...")
            for _ in tqdm(islice(kilt_it, scanned), total=scanned, desc="Fast-forward"):
                pass

    # ── 4. Scan loop with periodic checkpoints ────────────────────────────
    remaining = CONFIG['kilt_scan_limit'] - scanned
    for krow in tqdm(kilt_it, desc='Scanning KILT', total=remaining):
        if scanned >= CONFIG['kilt_scan_limit']:
            break
        scanned += 1

        norm_title = normalise_title(krow['wikipedia_title'])
        paras      = krow['text']['paragraph'] if isinstance(krow['text'], dict) else []
        paras      = [p for p in paras if isinstance(p, str) and len(p.strip()) > 50]
        if norm_title in target_norm2orig:
            kilt_pages[norm_title] = paras
        elif len(distractor_pool) < CONFIG['n_distractors'] and paras:
            distractor_pool.append({'title': krow['wikipedia_title'], 'paragraphs': paras})

        # Checkpoint every N records
        if scanned % CONFIG['checkpoint_every'] == 0:
            _ckpt = {'kilt_pages': kilt_pages, 'distractor_pool': distractor_pool,
                     'scanned': scanned}
            try:
                _ckpt['iter_state'] = kilt_it.state_dict()
            except Exception:
                pass  # state_dict not supported by this loader; fast-forward on resume
            with open(KILT_CHECKPOINT, 'wb') as f:
                pickle.dump(_ckpt, f)
            print(f"  ✓ Checkpoint at {scanned:,} ({len(kilt_pages)} pages found)")

    # ── 5. Save final cache, remove checkpoint ────────────────────────────
    with open(KILT_CACHE, 'wb') as f:
        pickle.dump({'kilt_pages': kilt_pages, 'distractor_pool': distractor_pool,
                     'scanned': scanned}, f)
    if os.path.exists(KILT_CHECKPOINT):
        os.remove(KILT_CHECKPOINT)
    print(f"\nScan complete. Cache saved -> {KILT_CACHE}")

print(f"\nScanned   : {scanned:,} / {CONFIG['kilt_scan_limit']:,}")
print(f"Found     : {len(kilt_pages)} / {len(target_norm2orig)} entity pages")
print(f"Distractors: {len(distractor_pool)}")
not_found_norms = set(target_norm2orig.keys()) - set(kilt_pages.keys())
print(f"Not reached in scan: {len(not_found_norms)}")
if not_found_norms:
    print("  e.g.", [target_norm2orig[k] for k in list(not_found_norms)[:5]])
print("\nNote: pages not reached are UNKNOWN (not confirmed negatives).")


Google Drive mounted — checkpoints/cache will survive runtime resets.
Target titles to find: 299

No cache or checkpoint — starting fresh scan (~15-30 min)...


Scanning KILT:   0%|          | 0/3000000 [00:00<?, ?it/s]

  ✓ Checkpoint at 200,000 (10 pages found)
  ✓ Checkpoint at 400,000 (26 pages found)
  ✓ Checkpoint at 600,000 (38 pages found)
  ✓ Checkpoint at 800,000 (49 pages found)
  ✓ Checkpoint at 1,000,000 (56 pages found)
  ✓ Checkpoint at 1,200,000 (66 pages found)
  ✓ Checkpoint at 1,400,000 (74 pages found)
  ✓ Checkpoint at 1,600,000 (81 pages found)
  ✓ Checkpoint at 1,800,000 (89 pages found)
  ✓ Checkpoint at 2,000,000 (96 pages found)
  ✓ Checkpoint at 2,200,000 (102 pages found)
  ✓ Checkpoint at 2,400,000 (114 pages found)
  ✓ Checkpoint at 2,600,000 (123 pages found)
  ✓ Checkpoint at 2,800,000 (127 pages found)
  ✓ Checkpoint at 3,000,000 (135 pages found)

Scan complete. Cache saved -> /content/drive/MyDrive/halt_rag_cache/kilt_cache_caf16b2f954a.pkl

Scanned   : 3,000,000 / 3,000,000
Found     : 135 / 299 entity pages
Distractors: 500
Not reached in scan: 164
  e.g. ['Smoke (Miscione novel)', 'Kowale, Lower Silesian Voivodeship', 'Kenneth Kennedy (bishop)', 'Liquid (musician)'

In [14]:
m2_match, m2_status = [], []
for _, row in df.iterrows():
    key = normalise_title(row['wiki_title'])   # same normalised key as kilt_pages
    if key not in kilt_pages:
        m2_match.append(np.nan)
        m2_status.append('not_in_kilt_scan')
        continue
    if not row['answers_list']:
        m2_match.append(np.nan)
        m2_status.append('no_valid_answers')
        continue
    body = normalise_text(' '.join(kilt_pages[key]))
    hit  = answer_in_text(row['answers_list'], body)
    m2_match.append(int(hit))
    m2_status.append('found' if hit else 'not_found')

df['m2_match']  = m2_match   # float: 1.0, 0.0, or NaN (unknown / unevaluable)
df['m2_status'] = m2_status
print("M2 done.")
print(df['m2_status'].value_counts().to_string())

M2 done.
m2_status
not_in_kilt_scan    164
found               120
not_found            16


In [15]:
print("=" * 65)
print("M2: KILT SUBJECT-PAGE MATCH (2019 snapshot)")
print("=" * 65)
for band in ['tail','head','overall']:
    sub    = df[df['band']==band] if band!='overall' else df
    found  = (sub['m2_status']=='found').sum()
    nf     = (sub['m2_status']=='not_found').sum()
    unk    = (sub['m2_status']=='not_in_kilt_scan').sum()
    nva    = (sub['m2_status']=='no_valid_answers').sum()
    # denominator = only rows where page was found AND answers exist
    evaluable = sub[sub['m2_status'].isin(['found','not_found'])]
    rate  = found / len(evaluable) if len(evaluable) > 0 else float('nan')
    print(f"  {band:7s}: {rate:.1%} match among {len(evaluable)} evaluable  "
          f"(found={found}  not_found={nf}  unknown={unk}  no_ans={nva})")

print()
print("Excluded from denominator: not_in_kilt_scan (page not reached), no_valid_answers.")

M2: KILT SUBJECT-PAGE MATCH (2019 snapshot)
  tail   : 82.8% match among 64 evaluable  (found=53  not_found=11  unknown=86  no_ans=0)
  head   : 93.1% match among 72 evaluable  (found=67  not_found=5  unknown=78  no_ans=0)
  overall: 88.2% match among 136 evaluable  (found=120  not_found=16  unknown=164  no_ans=0)

Excluded from denominator: not_in_kilt_scan (page not reached), no_valid_answers.


## M3 — Pilot retrieval@5
Chunk per-paragraph (not across paragraphs) to avoid splitting useful passages.
Distractor pages are convenience-sampled from first eligible KILT pages; label as pilot.


In [16]:
def chunk_text(paragraphs, chunk_words=CONFIG['chunk_words']):
    """Chunk each paragraph independently to preserve passage boundaries."""
    chunks = []
    for para in paragraphs:
        words = str(para).split()
        for i in range(0, len(words), chunk_words):
            chunk = ' '.join(words[i:i+chunk_words])
            if len(chunk.strip()) > 30:
                chunks.append(chunk)
    return chunks

passages = []
for orig, paras in kilt_pages.items():
    for chunk in chunk_text(paras):
        passages.append((orig, chunk))

for d in distractor_pool:
    for chunk in chunk_text(d['paragraphs']):
        passages.append((d['title'], chunk))

n_entity   = sum(1 for t,_ in passages if t in kilt_pages)
n_distract = len(passages) - n_entity
print(f"Pilot index: {len(passages):,} passages total")
print(f"  Entity pages   : {n_entity:,}")
print(f"  Distractors    : {n_distract:,}  (convenience-sampled, not random)")

Pilot index: 26,387 passages total
  Entity pages   : 2,835
  Distractors    : 23,552  (convenience-sampled, not random)


In [17]:
print("Encoding passages...")
model     = SentenceTransformer('all-MiniLM-L6-v2')
texts     = [p[1] for p in passages]
corp_embs = model.encode(texts, batch_size=256, show_progress_bar=True,
                          normalize_embeddings=True).astype('float32')

index = faiss.IndexFlatIP(corp_embs.shape[1])
index.add(corp_embs)
print(f"FAISS index: {index.ntotal:,} vectors  dim={corp_embs.shape[1]}")

Encoding passages...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/104 [00:00<?, ?it/s]

FAISS index: 26,387 vectors  dim=384


In [18]:
q_embs = model.encode(df['question'].tolist(), batch_size=256,
                       show_progress_bar=True,
                       normalize_embeddings=True).astype('float32')
_, I   = index.search(q_embs, CONFIG['top_k'])

m3_covered, m3_ret_titles = [], []
for qi, (_, row) in enumerate(df.iterrows()):
    ret_idx    = [i for i in I[qi] if i >= 0]
    ret_texts  = [texts[i] for i in ret_idx]
    ret_titles = list({passages[i][0] for i in ret_idx})
    combined   = normalise_text(' '.join(ret_texts))
    hit        = answer_in_text(row['answers_list'], combined)
    m3_covered.append(int(hit))
    m3_ret_titles.append(ret_titles)

df['m3_covered']    = m3_covered
df['m3_ret_titles'] = m3_ret_titles
print("M3 done.")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

M3 done.


In [19]:
print("=" * 65)
print(f"M3: PILOT RETRIEVAL@{CONFIG['top_k']} (small index, convenience-sampled)")
print("=" * 65)
for band in ['tail','head','overall']:
    sub  = df[df['band']==band] if band!='overall' else df
    cov  = sub['m3_covered'].mean()
    n    = len(sub)
    hits = sub['m3_covered'].sum()
    print(f"  {band:7s}: {cov:.1%}  ({hits}/{n})")

M3: PILOT RETRIEVAL@5 (small index, convenience-sampled)
  tail   : 43.3%  (65/150)
  head   : 38.7%  (58/150)
  overall: 41.0%  (123/300)


In [20]:
print("=" * 70)
print("COMBINED VALIDATION REPORT  (300-question pilot)")
print("=" * 70)
print(f"{'Metric':<42} {'Overall':>8} {'Tail':>8} {'Head':>8}")
print("-" * 70)

for label, col, exclude_unk in [
    ("M1 Subject-page match (Wikipedia)",   'm1_match',  False),
    ("M2 Subject-page match (KILT 2019)*",  'm2_match',  True),   # exclude NaN
    (f"M3 Pilot retrieval@{CONFIG['top_k']} (small index)", 'm3_covered', False),
]:
    rows = []
    for band in ['overall','tail','head']:
        sub = df[df['band']==band] if band!='overall' else df
        if exclude_unk:
            sub = sub[sub[col].notna()]
        v = sub[col].mean() if len(sub) else float('nan')
        rows.append(f"{v:>7.1%}" if not np.isnan(v) else "    n/a")
    print(f"{label:<42} {rows[0]} {rows[1]} {rows[2]}")

# Unknown count for M2
unk_all  = (df['m2_status']=='not_in_kilt_scan').sum()
unk_tail = (df[df['band']=='tail']['m2_status']=='not_in_kilt_scan').sum()
unk_head = (df[df['band']=='head']['m2_status']=='not_in_kilt_scan').sum()
print(f"  * M2 excludes {unk_all} unknown ({unk_tail} tail, {unk_head} head) from denominator")

print()
print("─" * 70)
# Conditional retrieval rate
m2_supported = df[df['m2_match'] == 1.0]
if len(m2_supported) > 0:
    cond_rate = m2_supported['m3_covered'].mean()
    print(f"Pilot answer-string retrieval rate among M2 matches: "
          f"{cond_rate:.1%}  (n={len(m2_supported)})")
    print(f"  → {(1-cond_rate):.1%} of M2-matched questions NOT retrieved (potential HALT-RAG signal)")
    print(f"    (Caution: retrieved text may come from distractor passages, not subject pages.)")
else:
    print("No M2 matches — cannot compute conditional retrieval rate.")

print()
print("Caveats:")
print("  M2 = subject-page match within scan window; cross-page evidence not measured.")
print("  M3 = pilot index; convenience-sampled distractors; not full-KILT retrieval.")
print("  Answer-string match ≠ semantic answerability. Manual audit required.")

COMBINED VALIDATION REPORT  (300-question pilot)
Metric                                      Overall     Tail     Head
----------------------------------------------------------------------
M1 Subject-page match (Wikipedia)            87.7%   80.7%   94.7%
M2 Subject-page match (KILT 2019)*           88.2%   82.8%   93.1%
M3 Pilot retrieval@5 (small index)           41.0%   43.3%   38.7%
  * M2 excludes 164 unknown (86 tail, 78 head) from denominator

──────────────────────────────────────────────────────────────────────
Pilot answer-string retrieval rate among M2 matches: 88.3%  (n=120)
  → 11.7% of M2-matched questions NOT retrieved (potential HALT-RAG signal)
    (Caution: retrieved text may come from distractor passages, not subject pages.)

Caveats:
  M2 = subject-page match within scan window; cross-page evidence not measured.
  M3 = pilot index; convenience-sampled distractors; not full-KILT retrieval.
  Answer-string match ≠ semantic answerability. Manual audit required.


In [21]:
import datasets as hf_datasets

meta = {
    **CONFIG,
    'datasets_version': hf_datasets.__version__,
    'n_kilt_pages_found': len(kilt_pages),
    'n_kilt_pages_unknown': int((df['m2_status']=='not_in_kilt_scan').sum()),
    'pilot_index_size': len(passages),
}
with open('validation_v4_config.json', 'w') as f:
    json.dump(meta, f, indent=2)

df.drop(columns=['m3_ret_titles']).to_csv('popqa_validated_v4.csv', index=False)

print("Saved popqa_validated_v4.csv")
print("Saved validation_v4_config.json")
print()
print("Config snapshot:")
print(json.dumps(meta, indent=2))

Saved popqa_validated_v4.csv
Saved validation_v4_config.json

Config snapshot:
{
  "n_tail": 150,
  "n_head": 150,
  "tail_max_pop": 50,
  "head_min_pop": 500,
  "seed": 42,
  "top_k": 5,
  "chunk_words": 100,
  "n_distractors": 500,
  "kilt_scan_limit": 3000000,
  "checkpoint_every": 200000,
  "min_ans_chars": 2,
  "run_timestamp": "2026-10-24T17:58:55.228634Z",
  "datasets_version": "2.21.0",
  "n_kilt_pages_found": 135,
  "n_kilt_pages_unknown": 164,
  "pilot_index_size": 26387
}


In [22]:
# ── Spot-check 1: M2 match but M3 miss  (retrieval failure cases) ─────────────
print("=== M2=found, M3=0 (retrieval failures) ===")
fail = df[(df['m2_match']==1.0) & (df['m3_covered']==0)].head(4)
for _, r in fail.iterrows():
    print(f"  [{r['band']}] Q  : {r['question']}")
    print(f"         ans : {r['answers_list'][:2]}")
    print(f"         title: {r['wiki_title']}")
    print()

# ── Spot-check 2: M3 hit but M2=0/unknown (distractor match or scan miss) ────
print("=== M3=1, M2≠found (possible distractor hit or scan miss) ===")
dist_hit = df[(df['m3_covered']==1) & (df['m2_match']!=1.0)].head(3)
for _, r in dist_hit.iterrows():
    print(f"  [{r['band']}] Q  : {r['question']}")
    print(f"         m2_status: {r['m2_status']}")
    print(f"         retrieved from: {r['m3_ret_titles']}")
    print()

=== M2=found, M3=0 (retrieval failures) ===
  [tail] Q  : Who is the author of Follow The Music?
         ans : ['Jac Holzman', 'Jacob Holzman']
         title: Follow the Music

  [tail] Q  : In what city was Jiří Vyčichlo born?
         ans : ['Prague', 'Praha']
         title: Jiří Vyčichlo

  [tail] Q  : Who is the author of Looking Forward?
         ans : ['Stephen Marlowe']
         title: Looking Forward (anthology)

  [tail] Q  : In what country is Mary?
         ans : ['France', 'fr']
         title: Mary, Saône-et-Loire

=== M3=1, M2≠found (possible distractor hit or scan miss) ===
  [tail] Q  : What sport does Lukáš Bodeček play?
         m2_status: not_in_kilt_scan
         retrieved from: ['Tychy', 'Gliwice']

  [tail] Q  : In what country is Gornji Davidovići?
         m2_status: not_in_kilt_scan
         retrieved from: ['Bytom', 'Gliwice', 'Kielce', 'miroslav milutinović']

  [tail] Q  : What sport does Wojciech Jarmuż play?
         m2_status: not_in_kilt_scan
        

## Manual audit checklist
- [ ] 5 M2=found examples — confirm KILT passage actually contains the answer
- [ ] 5 M2=not_found examples — confirm answer really absent (not normalisation miss)
- [ ] 5 not_in_kilt_scan titles — search manually to see if they exist late in KILT
- [ ] 3 M3=1 / M2≠found rows — check if answer came from distractor passage
- [ ] Confirm both m3_covered=0 and m3_covered=1 exist in **both** tail and head
- [ ] Record all caveats when writing up results
